# baostock 数据获取

本 notebook 演示如何用 [baostock](http://baostock.com) 拉取 A 股指数日线数据并落盘到 `exports/`，包含：

1. **单标的示例**：上证综指 `sh.000001`
2. **批量拉取**：沪深 300 / 中证 500 / 上证 50（供后续 market_regime 分析）

> baostock 在同一进程内只需要 login 一次，即可连续发起多次查询；末尾统一 logout。

## 1. 登录

In [ ]:
from pathlib import Path
import baostock as bs
import pandas as pd
from datetime import datetime

lg = bs.login()
assert lg.error_code == '0', f"登录失败: {lg.error_msg}"
print(f"登录: {lg.error_code} - {lg.error_msg}")

## 2. 项目路径与配置

定位项目根（找 `.gitignore` 作为标志），所有 CSV 统一输出到 `exports/`。

**指数清单**：

| 代码 | baostock | 名称 | 用途 |
|---|---|---|---|
| 000300 | sh.000300 | 沪深 300 | 主基准 |
| 000905 | sh.000905 | 中证 500 | 中盘，与 300 互补 |
| 000016 | sh.000016 | 上证 50 | 大盘，与 300 互补 |

时间范围：`2010-01-01 ~ 今日`，覆盖 2015 疯牛、2018 贸易战、2020 疫情、2022 熔断等多轮 regime。

In [ ]:
# 定位项目根目录（向上查找 .gitignore）
project_root = Path.cwd()
while not (project_root / '.gitignore').exists():
    if project_root.parent == project_root:
        raise RuntimeError('找不到项目根')
    project_root = project_root.parent

# 输出目录（统一）
EXPORT_DIR = project_root / 'exports'
EXPORT_DIR.mkdir(exist_ok=True)
print(f'项目根: {project_root}')
print(f'输出目录: {EXPORT_DIR}')

# 时间范围（如需更换改这里）
START_DATE = '2010-01-01'
END_DATE = datetime.now().strftime('%Y-%m-%d')
print(f'时间范围: {START_DATE} ~ {END_DATE}')

# 指数清单：(baostock 代码, 名称, 输出文件名)
INDICES = [
    ('sh.000300', '沪深300', '000300'),
    ('sh.000905', '中证500', '000905'),
    ('sh.000016', '上证50',  '000016'),
]

## 3. 单标的示例：上证综指

展示最朴素的「查一个标的 → 写 CSV」流程。

In [ ]:
rs = bs.query_history_k_data_plus(
    'sh.000001',
    fields='date,open,high,low,close,volume,amount',
    start_date='2020-01-01',
    end_date=END_DATE,
    frequency='d',
    adjustflag='2',  # 前复权
)
assert rs.error_code == '0', f'查询失败: {rs.error_msg}'

rows = []
while rs.next():
    rows.append(rs.get_row_data())

df = pd.DataFrame(rows, columns=rs.fields)
out_path = EXPORT_DIR / '上证综指_日线.csv'
df.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'上证综指: {len(df)} 行 → {out_path}')
df.head()

## 4. 批量拉取：沪深 300 / 中证 500 / 上证 50

封装一个 `fetch_one` 函数，循环遍历 `INDICES`，手写 CSV 表头避免 pandas dtype 推断。

In [ ]:
def fetch_one(bs_code: str, name: str, out_code: str) -> Path:
    """拉取一个指数并写入 CSV，返回输出路径。"""
    rs = bs.query_history_k_data_plus(
        bs_code,
        'date,open,high,low,close,volume',
        start_date=START_DATE,
        end_date=END_DATE,
        frequency='d',
        adjustflag='2',  # 前复权
    )
    if rs.error_code != '0':
        raise RuntimeError(f'{bs_code} 查询失败: {rs.error_msg}')

    rows = []
    while rs.next():
        rows.append(rs.get_row_data())
    if not rows:
        raise RuntimeError(f'{bs_code} 无数据')

    out = EXPORT_DIR / f'{out_code}.csv'
    with open(out, 'w', encoding='utf-8') as f:
        f.write('date,open,high,low,close,volume\n')
        for r in rows:
            f.write(','.join(r) + '\n')
    print(f'  {bs_code} {name} → {out.name}: {len(rows)} 行（{rows[0][0]} ~ {rows[-1][0]}）')
    return out

for bs_code, name, out_code in INDICES:
    fetch_one(bs_code, name, out_code)

## 5. 验证输出

抽查一个 CSV 的头尾，确认写入正确。

In [ ]:
sample = pd.read_csv(EXPORT_DIR / '000300.csv')
print(f'沪深 300 CSV: {len(sample)} 行')
print(f'  头: {sample.iloc[0].to_dict()}')
print(f'  尾: {sample.iloc[-1].to_dict()}')

# 三个文件应同时存在
for code in ['000300', '000905', '000016']:
    p = EXPORT_DIR / f'{code}.csv'
    assert p.exists(), f'{p} 不存在'
    print(f'  ✓ {p.name} ({p.stat().st_size} bytes)')

## 6. 退出登录

In [ ]:
bs.logout()
print('已退出 baostock')

## 7. 刷新数据

需要更新数据时：Run All 即可，会覆盖 `exports/` 下的 CSV。

如需更换标的或时间范围，修改上面的 `INDICES` / `START_DATE` / `END_DATE`。